# Connect

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/KLTN

Mounted at /content/drive
/content/drive/MyDrive/KLTN


In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.7 MB/s eta 0:00:00


# Load dữ liệu

In [ ]:
import pandas as pd
import numpy as np
from numpy import savetxt, loadtxt

def load_dataset(label_count):
    # Tải dữ liệu từ file CSV
    data = pd.read_csv("/content/drive/MyDrive/KLTN/DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0', axis=1)

    # Các cột nhãn
    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Lấy số lượng hàng dữ liệu là 39645
    data = data.iloc[:39645]

    # Chia tập dữ liệu thành tập huấn luyện và tập kiểm tra
    total_rows = len(data)
    split_point = int(0.8 * total_rows)
    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    features_source_train =  np.loadtxt("/content/drive/MyDrive/KLTN/DATASET/split_train.csv", delimiter=',')
    features_source_test = np.loadtxt("/content/drive/MyDrive/KLTN/DATASET/split_test.csv", delimiter=',')
    X_train_source = np.array(features_source_train)
    X_test_source = np.array(features_source_test)
    print("Load Features codeBERT success!!")
    # Lấy các đặc trưng từ tập dữ liệu
    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # Lấy nhãn từ tập dữ liệu
    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")
    # Load GNN

    # Load data
    input_file = '/content/drive/MyDrive/KLTN/DATASET/gnn_input_binary_labels.pkl'

    with open(input_file, 'rb') as f:
        data = pickle.load(f)

    loaded_feature_matrices = data['features']
    adj_matrices = data['adj_matrices']
    # loaded_all_labels = data['labels']
    # Define the target dimension
    target_dim = 6000

    def pad_or_truncate_features(features, target_dim):
        if features.shape[1] > target_dim:
            return features[:, :target_dim]
        elif features.shape[1] < target_dim:
            padded_features = np.zeros((features.shape[0], target_dim))
            padded_features[:, :features.shape[1]] = features
            return padded_features
        else:
            return features

    # Function to create PyTorch Geometric Data object from feature matrices and labels
    def create_pyg_data(features, adj_matrix, labels, target_dim):
        num_nodes = features.shape[0]
        edge_index = torch.tensor(np.array(adj_matrix), dtype=torch.long).nonzero(as_tuple=False).t().contiguous()
        x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
        y = torch.tensor(labels, dtype=torch.float)  # Ensure labels are float for BCEWithLogitsLoss
        data = Data(x=x, edge_index=edge_index, y=y)
        return data

    data_list = [create_pyg_data(features, adj_matrix, labels, target_dim) for features, adj_matrix, labels in zip(loaded_feature_matrices, adj_matrices, labels)]

    # Create DataLoader
    train_loader = DataLoader(data_list[:int(len(data_list)*0.8)], batch_size=32, shuffle=True)  # Adjust batch size as needed
    test_loader = DataLoader(data_list[int(len(data_list)*0.8):], batch_size=32, shuffle=False)  # Adjust batch size as needed


    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, train_loader, test_loader, y_train, y_test, labels



In [ ]:
X_train_opcode_tensor, X_test_opcode_tensor, X_train_source_tensor, X_test_source_tensor, train_loader, test_loader, y_train_tensor, y_test_tensor, labels = load_dataset(8)


Load Features codeBERT success!!
Load Data for 8-Label MultiLabel success!!


In [ ]:
import torch
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
import pickle

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = x.relu()
        x = global_mean_pool(x, batch)  # Pooling layer
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        x = torch.sigmoid(x)  # Sigmoid activation for multi-label classification
        return x

# Load mô hình GNN
model_path = '/content/drive/MyDrive/KLTN/Unimodal/GCN_60E.pth'

# Adjust the in_channels, hidden_channels, and out_channels parameters based on the training settings
gnn_model = GCN(in_channels=6000, hidden_channels=64, out_channels=8)
gnn_model.load_state_dict(torch.load(model_path))
gnn_model.eval()  # Set the model to evaluation mode

# Function to get GNN outputs for a given DataLoader
def get_gnn_outputs(loader):
    outputs = []
    with torch.no_grad():
        for data in loader:
            out = gnn_model(data)
            outputs.append(out.numpy())
    return np.concatenate(outputs)

# Get GNN outputs for training and testing data
gnn_train_outputs = get_gnn_outputs(train_loader)
gnn_test_outputs = get_gnn_outputs(test_loader)

In [ ]:
import numpy
gnn_test_outputs = numpy.array(gnn_test_outputs)
gnn_train_outputs = numpy.array(gnn_train_outputs)


# Khởi tạo model

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Input, concatenate, Reshape, Conv1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
bert_model = load_model('/content/drive/MyDrive/KLTN/Unimodal/CodeBERT_30E.h5', compile=False)
lstm_model = load_model('/content/drive/MyDrive/KLTN/Unimodal/BiLSTM_Full_2gram_Softmax_30E.h5', compile=False)

# Tắt đi trainable của hai mô hình
bert_model.trainable = False
lstm_model.trainable = False
bert_output = bert_model.output
lstm_output = lstm_model.output
gnn_output = tf.keras.layers.Input(shape=(8,), name="gnn_output")  # Kích thước đầu ra của GNN

# Tắt trainable cho GNN
gnn_output._keras_history[0].trainable = False

In [ ]:
concatenated = concatenate([bert_output, lstm_output, gnn_output], axis=-1)
concatenated_reshaped = Reshape((24, 1))(concatenated)
conv_out = Conv1D(64, 3, activation='relu')(concatenated_reshaped)
flatten_out = Flatten()(conv_out)
dense_features = Dense(128, activation='relu', name='dense_multilabel')(flatten_out)
dense_features_2 = Dropout(0.3, name='drop_multilabel')(dense_features)
final_out = Dense(8, activation='sigmoid', name='output_multimodal')(dense_features_2)

In [ ]:
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]
# Tạo mô hình tổng cộng
model = tf.keras.models.Model(inputs=[bert_model.input, lstm_model.input, gnn_output], outputs=final_out)

# Compile mô hình
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

In [ ]:

history = model.fit(
    [X_train_source_tensor, X_train_opcode_tensor, gnn_train_outputs],
    y_train_tensor,
    validation_data=([X_test_source_tensor, X_test_opcode_tensor, gnn_test_outputs], y_test_tensor),
    epochs=20,
    batch_size=32
)

Epoch 1/20
992/992 [==============================] - 21s 14ms/step - loss: 0.2768 - accuracy: 0.8898 - precision: 0.8521 - recall: 0.8103 - val_loss: 0.1752 - val_accuracy: 0.9381 - val_precision: 0.9035 - val_recall: 0.8898
Epoch 2/20
992/992 [==============================] - 13s 13ms/step - loss: 0.2504 - accuracy: 0.8996 - precision: 0.8590 - recall: 0.8363 - val_loss: 0.1722 - val_accuracy: 0.9376 - val_precision: 0.8998 - val_recall: 0.8927
Epoch 3/20
992/992 [==============================] - 13s 13ms/step - loss: 0.2485 - accuracy: 0.9003 - precision: 0.8619 - recall: 0.8348 - val_loss: 0.1677 - val_accuracy: 0.9386 - val_precision: 0.9131 - val_recall: 0.8804
Epoch 4/20
992/992 [==============================] - 13s 13ms/step - loss: 0.2471 - accuracy: 0.9004 - precision: 0.8641 - recall: 0.8321 - val_loss: 0.1641 - val_accuracy: 0.9390 - val_precision: 0.9104 - val_recall: 0.8849
Epoch 5/20
992/992 [==============================] - 13s 13ms/step - loss: 0.2454 - accuracy: 0

In [ ]:
def apply_threshold(y_pred_prob, threshold=0.5):
    y_pred_prob = np.array(y_pred_prob)
    y_pred_thresh = np.zeros_like(y_pred_prob)
    y_pred_thresh[y_pred_prob >= threshold] = 1
    return y_pred_thresh

In [ ]:
y_pred_prob = model.predict([X_test_source_tensor, X_test_opcode_tensor, gnn_test_outputs])
y_pred = apply_threshold(y_pred_prob)

248/248 [==============================] - 3s 9ms/step


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
label_names = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

for i, label in enumerate(label_names):
    precision = precision_score(y_test_tensor[:, i], y_pred[:, i])
    recall = recall_score(y_test_tensor[:, i], y_pred[:, i])
    f1 = f1_score(y_test_tensor[:, i], y_pred[:, i])
    accuracy = accuracy_score(y_test_tensor[:, i], y_pred[:, i])
    # Tính confusion matrix cho nhãn i
    tn, fp, fn, tp = confusion_matrix(y_test_tensor[:, i], y_pred[:, i]).ravel()

    # Tính FPR và FNR
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    print(f"Metrics for {label}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  FPR:       {fpr:.4f}")
    print(f"  FNR:       {fnr:.4f}")
    print()

Metrics for Other:
  Accuracy:  0.8794
  Precision: 0.8763
  Recall:    0.9161
  F1-Score:  0.8957
  FPR:       0.1683
  FNR:       0.0839

Metrics for access_control:
  Accuracy:  0.9580
  Precision: 0.9183
  Recall:    0.5529
  F1-Score:  0.6902
  FPR:       0.0045
  FNR:       0.4471

Metrics for arithmetic:
  Accuracy:  0.9506
  Precision: 0.9640
  Recall:    0.9713
  F1-Score:  0.9676
  FPR:       0.1155
  FNR:       0.0287

Metrics for denial_service:
  Accuracy:  0.9654
  Precision: 0.9012
  Recall:    0.8986
  F1-Score:  0.8999
  FPR:       0.0206
  FNR:       0.1014

Metrics for front_running:
  Accuracy:  0.9550
  Precision: 0.8864
  Recall:    0.7980
  F1-Score:  0.8398
  FPR:       0.0178
  FNR:       0.2020

Metrics for reentrancy:
  Accuracy:  0.9097
  Precision: 0.8756
  Recall:    0.8197
  F1-Score:  0.8467
  FPR:       0.0509
  FNR:       0.1803

Metrics for time_manipulation:
  Accuracy:  0.9615
  Precision: 0.8821
  Recall:    0.4759
  F1-Score:  0.6183
  FPR:       